# 01 — Data Exploration
**Health Risk Triage AI — Phase 1**

This notebook explores the raw PhysioNet 2019 Sepsis Challenge dataset after it has been parsed by `src/data_loading.py`. The goal is to understand the dataset's structure, missingness patterns, vital-sign distributions, and class balance **before** any modelling decisions are made.

All observations here directly informed the design of `labeling.py`, `features.py`, and `preprocessing.py`.

---
**Pipeline position:** `data_loading.py` → **[YOU ARE HERE]** → `labeling.py`

**Input:** `data/processed/phase1/phase1_raw.parquet`  
**Output:** Observations only — no files written by this notebook

## 0. Imports and Config

In [ ]:
import sys
sys.path.append('..')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Load central config
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

DATA_PATH = '../data/processed/phase1/phase1_raw.parquet'
VITALS    = config['features']['vitals']

pd.set_option('display.float_format', '{:.3f}'.format)
print('Config loaded. Vitals in scope:', VITALS)

## 1. Load the Dataset

In [ ]:
df = pd.read_parquet(DATA_PATH)

print(f'Shape          : {df.shape}')
print(f'Unique patients: {df["PatientID"].nunique():,}')
print(f'Columns        : {list(df.columns)}')
df.head()

## 2. ICU Stay Duration

Each row is one patient-hour. `ICULOS` is the hour counter per patient.

In [ ]:
stay_hours = df.groupby('PatientID')['ICULOS'].max()
print('ICU stay duration (hours):')
print(stay_hours.describe().round(1))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(stay_hours, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(stay_hours.median(), color='tomato', linestyle='--', label=f'Median: {stay_hours.median():.0f}h')
ax.set_xlabel('ICU Stay Duration (hours)')
ax.set_ylabel('Number of Patients')
ax.set_title('Distribution of ICU Stay Duration')
ax.legend()
plt.tight_layout()
plt.show()

## 3. SepsisLabel — Row-Level vs Patient-Level

Key insight: the row-level positive rate (~2%) is much lower than the patient-level rate (~9%). This is because most of a septic patient's ICU hours occur **before** the 6-hour pre-onset window where `SepsisLabel=1`.

In [ ]:
row_rate     = df['SepsisLabel'].mean()
patient_rate = df.groupby('PatientID')['SepsisLabel'].max().mean()

print(f'Row-level  positive rate : {row_rate:.2%}  ({df["SepsisLabel"].sum():,} / {len(df):,} rows)')
print(f'Patient-level sepsis rate: {patient_rate:.2%}  ({df.groupby("PatientID")["SepsisLabel"].max().sum():.0f} / {df["PatientID"].nunique()} patients)')
print()
print('Implication: row-level imbalance is extreme (~97% negative).')
print('Class weighting and F1-macro (not accuracy) will be essential for fair evaluation.')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Row-level
axes[0].bar(['No Sepsis\n(SepsisLabel=0)', 'Sepsis\n(SepsisLabel=1)'],
            [1 - row_rate, row_rate],
            color=['steelblue', 'tomato'])
axes[0].set_title('Row-Level Label Distribution')
axes[0].set_ylabel('Proportion of rows')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# Patient-level
axes[1].bar(['Never Septic', 'Develops Sepsis'],
            [1 - patient_rate, patient_rate],
            color=['steelblue', 'tomato'])
axes[1].set_title('Patient-Level Sepsis Rate')
axes[1].set_ylabel('Proportion of patients')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.suptitle('SepsisLabel: Row-Level vs Patient-Level', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Missingness Analysis

PhysioNet vitals are **intermittently measured** — not every vital is recorded every hour. Understanding missingness per vital directly informed our 3-tier imputation strategy in `preprocessing.py`.

In [ ]:
miss = df[VITALS].isna().mean().sort_values(ascending=False) * 100
print('Missingness by vital (%):')
print(miss.round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(miss.index, miss.values,
               color=['tomato' if v > 50 else 'steelblue' for v in miss.values])
ax.axvline(50, color='tomato', linestyle='--', alpha=0.5, label='50% threshold')
ax.set_xlabel('Missing (%)')
ax.set_title('Vital Sign Missingness')
ax.legend()
for bar, val in zip(bars, miss.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nDesign note: Temp (>50% missing) required 3-tier imputation.')
print('DBP (48% missing) was partly recovered algebraically from SBP+MAP in features.py.')

## 5. Vital Sign Distributions

Distribution of each vital, split by SepsisLabel, to verify the data makes clinical sense before modelling.

In [ ]:
vitals_to_plot = ['HR', 'Resp', 'O2Sat', 'SBP', 'Temp']

# Normal range reference lines (NEWS2 boundaries)
normal_ranges = {
    'HR'   : (51, 90),
    'Resp' : (12, 20),
    'O2Sat': (96, 100),
    'SBP'  : (111, 219),
    'Temp' : (36.1, 38.0),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, vital in zip(axes, vitals_to_plot):
    no_sep = df.loc[df['SepsisLabel'] == 0, vital].dropna()
    sep    = df.loc[df['SepsisLabel'] == 1, vital].dropna()

    ax.hist(no_sep, bins=50, alpha=0.6, color='steelblue',
            density=True, label='No Sepsis')
    ax.hist(sep,    bins=50, alpha=0.7, color='tomato',
            density=True, label='Sepsis')

    if vital in normal_ranges:
        lo, hi = normal_ranges[vital]
        ax.axvline(lo, color='green', linestyle=':', linewidth=1.2, alpha=0.8)
        ax.axvline(hi, color='green', linestyle=':', linewidth=1.2, alpha=0.8)

    ax.set_title(vital)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    if vital == 'HR':
        ax.legend(fontsize=8)

plt.suptitle('Vital Distributions: Sepsis vs Non-Sepsis (green lines = NEWS2 normal range)',
             fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Age and Demographics

In [ ]:
# Age is constant per patient — take first row per patient
patient_df = df.groupby('PatientID').first().reset_index()

print(f"Age range  : {patient_df['Age'].min():.1f} - {patient_df['Age'].max():.1f} years")
print(f"Age median : {patient_df['Age'].median():.1f} years")
print(f"Gender     : {patient_df['Gender'].value_counts().to_dict()}  (0=Female, 1=Male)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Age distribution by sepsis status
sep_patients    = patient_df[patient_df['SepsisLabel'] == 1]['Age']
nonsep_patients = patient_df[patient_df['SepsisLabel'] == 0]['Age']

axes[0].hist(nonsep_patients, bins=40, alpha=0.6, color='steelblue',
             density=True, label='No Sepsis')
axes[0].hist(sep_patients,    bins=40, alpha=0.7, color='tomato',
             density=True, label='Sepsis')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Density')
axes[0].set_title('Age Distribution by Sepsis Status')
axes[0].legend()

# Gender split
gender_counts = patient_df['Gender'].value_counts()
axes[1].bar(['Female (0)', 'Male (1)'],
            [gender_counts.get(0, 0), gender_counts.get(1, 0)],
            color=['mediumorchid', 'steelblue'])
axes[1].set_ylabel('Number of Patients')
axes[1].set_title('Gender Distribution')

plt.suptitle('Patient Demographics', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Blood Pressure Trio — Co-Missingness

SBP, MAP, and DBP are mathematically related: `MAP ≈ DBP + (SBP-DBP)/3`. This means where SBP and MAP are present but DBP is missing, we can **derive** DBP algebraically rather than imputing it. This analysis motivated the algebraic recovery step in `features.py`.

In [ ]:
bp = df[['SBP', 'MAP', 'DBP']].isna()

derivable     = ((~bp['SBP']) & (~bp['MAP']) &   bp['DBP']).sum()
unrecoverable = (  bp['SBP']  &   bp['MAP']  &   bp['DBP']).sum()
present       = (~bp['DBP']).sum()
total         = len(df)

print(f'DBP already present              : {present:>7,}  ({present/total:.1%})')
print(f'DBP derivable from SBP+MAP       : {derivable:>7,}  ({derivable/total:.1%})')
print(f'All three BP missing (fallback)  : {unrecoverable:>7,}  ({unrecoverable/total:.1%})')
print(f'Total rows                       : {total:>7,}')
print()
print(f'Algebraic recovery covers {derivable/(total-present):.1%} of missing DBP rows.')

fig, ax = plt.subplots(figsize=(7, 4))
labels = ['Present', 'Derivable\n(SBP+MAP available)', 'Unrecoverable\n(all missing)']
sizes  = [present, derivable, unrecoverable]
colors = ['steelblue', 'mediumseagreen', 'tomato']
ax.bar(labels, sizes, color=colors, edgecolor='white')
ax.set_ylabel('Row count')
ax.set_title('DBP Missingness — Recovery Potential')
for bar, val in zip(ax.patches, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 8. Correlation Between Vitals

In [ ]:
corr = df[VITALS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)

ax.set_xticks(range(len(VITALS)))
ax.set_yticks(range(len(VITALS)))
ax.set_xticklabels(VITALS, rotation=45, ha='right')
ax.set_yticklabels(VITALS)

for i in range(len(VITALS)):
    for j in range(len(VITALS)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')

ax.set_title('Vital Sign Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

print('SBP-MAP correlation is high (expected — MAP is derived from SBP+DBP).')
print('This is why all three BP columns are retained: MAP adds signal beyond SBP alone.')

## 9. Summary of Key Findings

| Finding | Value | Implication |
|---|---|---|
| Row-level sepsis rate | ~2.2% | Extreme class imbalance — accuracy is a useless metric |
| Patient-level sepsis rate | ~9.0% | 1 in 11 ICU patients develop sepsis |
| Avg ICU stay | ~39 hours | Time-series vitals are available per patient |
| Temp missingness | ~65% | Highest gap — requires forward-fill + median fallback |
| DBP derivable rows | ~31.6% | Algebraic recovery from SBP+MAP preferred over imputation |
| SBP-MAP correlation | High | Expected — supports keeping both as features |
| Sepsis shifts vitals | Yes | Tachycardia, low SpO2 visible in Sepsis distribution |

These findings directly shaped the decisions in `labeling.py`, `features.py`, and `preprocessing.py`.

**Next notebook:** `02_preprocessing.ipynb` — imputation, scaling, and class balance after the full pipeline.